# 01 — PaySim data audit

This notebook is the first step of the fraud-risk project. We will inspect the downloaded CSV before training anything. The audit checks its shape, columns, missing values, transaction types, label balance, time coverage, and obvious leakage risks.

> **Important:** PaySim is simulated data. A model score is only a signal for human review, never proof of wrongdoing.

## Before running
1. Download the PaySim dataset from its original provider and review the provider's terms.
2. Place the CSV in `data/raw/`.
3. In VS Code, select the `Python (fraud-graph)` notebook kernel.
4. Run each cell from top to bottom.

In [31]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print(f"Project folder: {PROJECT_ROOT}")
print(f"Raw data folder: {RAW_DATA_DIR}")
print(f"CSV files found: {[p.name for p in csv_files]}")

Project folder: c:\Users\REVANTH\.codex\.chatgpt-projects\g-p-6ad4704ed5b08191a0f999b6f2c61ad6\fraud-graph
Raw data folder: c:\Users\REVANTH\.codex\.chatgpt-projects\g-p-6ad4704ed5b08191a0f999b6f2c61ad6\fraud-graph\data\raw
CSV files found: ['PS_20174392719_1491204439457_log.csv']


If no CSV is listed above, stop here and put the downloaded PaySim CSV in `data/raw/`. Do not upload the raw dataset to GitHub.

In [32]:
if not csv_files:
    raise FileNotFoundError(
        f"No CSV found in {RAW_DATA_DIR}. Download PaySim and place its CSV there."
    )

data_path = csv_files[0]
print(f"Reading: {data_path.name}")
df = pd.read_csv(data_path, nrows=100_000)
print(f"Loaded sample shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Reading: PS_20174392719_1491204439457_log.csv
Loaded sample shape: 100,000 rows × 11 columns


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## First checks
We start with a sample to keep the notebook responsive. Once the file's columns and format are confirmed, the next step will profile the full dataset and build a time-aware validation plan.

In [33]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values in sample:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nNumeric summary:")
display(df.describe(include="all").T)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Data types:


,dtype
step,int64
type,object
amount,float64
nameOrig,object
oldbalanceOrg,float64
newbalanceOrig,float64
nameDest,object
oldbalanceDest,float64
newbalanceDest,float64
isFraud,int64



Missing values in sample:


,missing
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0



Numeric summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
step,100000.0,NaN,NaN,NaN,8.49964,1.825545,1.0,8.0,9.0,10.0,10.0
type,100000,5,PAYMENT,39512,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,100000.0,NaN,NaN,NaN,173602.221764,344300.32009,0.32,9963.5625,52745.52,211763.15,10000000.0
nameOrig,100000,100000,C1231006815,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceOrg,100000.0,NaN,NaN,NaN,877757.513394,2673284.413047,0.0,0.0,20061.5,190192.025,33797391.55
newbalanceOrig,100000.0,NaN,NaN,NaN,894061.916887,2711317.815613,0.0,0.0,0.0,214813.25,34008736.98
nameDest,100000,51551,C985934102,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceDest,100000.0,NaN,NaN,NaN,880504.806219,2402267.213443,0.0,0.0,20839.43,588272.375,34008736.98
newbalanceDest,100000.0,NaN,NaN,NaN,1184041.424221,2802349.681241,0.0,0.0,49909.18,1058186.155,38946233.02
isFraud,100000.0,NaN,NaN,NaN,0.00116,0.034039,0.0,0.0,0.0,0.0,1.0


## Full-file audit (chunked)
The earlier preview read only the first 100,000 rows. This cell scans the complete CSV in batches of 500,000 rows, so we can measure the actual fraud rate, missing values, time-step coverage, and transaction types without keeping the whole file in memory.

In [7]:
from collections import Counter
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
raw_data_dir = project_root / "data" / "raw"
csv_files = sorted(raw_data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No PaySim CSV found in {raw_data_dir}")
data_path = csv_files[0]
print(f"Auditing: {data_path.name}")

audit_columns = [
    "step", "type", "amount", "nameOrig", "nameDest",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest",
    "isFraud", "isFlaggedFraud",
]
rows_scanned = 0
fraud_count = 0
flagged_count = 0
min_step = None
max_step = None
type_counts = Counter()
type_fraud_counts = Counter()
label_counts = Counter()
flag_label_counts = Counter()
fraud_step_counts = Counter()
missing_counts = pd.Series(0, index=audit_columns, dtype="int64")

for batch_number, chunk in enumerate(
    pd.read_csv(data_path, usecols=audit_columns, chunksize=500_000), start=1
):
    rows_scanned += len(chunk)
    fraud_count += int(chunk["isFraud"].sum())
    flagged_count += int(chunk["isFlaggedFraud"].sum())
    min_step = int(chunk["step"].min()) if min_step is None else min(min_step, int(chunk["step"].min()))
    max_step = int(chunk["step"].max()) if max_step is None else max(max_step, int(chunk["step"].max()))
    type_counts.update(chunk["type"].value_counts().to_dict())
    fraud_by_type = chunk.loc[chunk["isFraud"] == 1].groupby("type").size()
    type_fraud_counts.update(fraud_by_type.to_dict())
    label_counts.update(chunk["isFraud"].value_counts().to_dict())
    flag_vs_label = chunk.groupby(["isFlaggedFraud", "isFraud"]).size()
    flag_label_counts.update({(int(flag), int(label)): int(count) for (flag, label), count in flag_vs_label.items()})
    fraud_by_step = chunk.loc[chunk["isFraud"] == 1].groupby("step").size()
    fraud_step_counts.update(fraud_by_step.to_dict())
    missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0).astype("int64")
    print(f"Scanned batch {batch_number}: {rows_scanned:,} rows so far")

print(f"\nTotal transactions: {rows_scanned:,}")
print(f"Fraud transactions: {fraud_count:,} ({fraud_count / rows_scanned:.4%})")
print(f"System-flagged transactions: {flagged_count:,}")
print(f"Step range: {min_step} to {max_step}")
print(f"Fraud labels contain only 0 and 1: {set(label_counts).issubset({0, 1})}")
print(f"Time steps containing fraud: {len(fraud_step_counts):,}")
print("\nFraud-label counts:")
display(pd.Series(label_counts, name="transactions").rename_axis("isFraud").to_frame())
flag_label_table = pd.DataFrame(
    [{"isFlaggedFraud": flag, "isFraud": label, "transactions": count}
     for (flag, label), count in sorted(flag_label_counts.items())]
)
print("\nExisting system flag versus dataset fraud label:")
display(flag_label_table)

type_summary = pd.DataFrame({
    "transactions": pd.Series(type_counts),
    "fraud_transactions": pd.Series(type_fraud_counts),
}).fillna(0).astype("int64")
type_summary["fraud_rate_percent"] = (
    type_summary["fraud_transactions"] / type_summary["transactions"] * 100
)
display(type_summary.sort_values("transactions", ascending=False))
print("\nMissing values across the full file:")
display(missing_counts.to_frame("missing_values"))

Auditing: PS_20174392719_1491204439457_log.csv
Scanned batch 1: 500,000 rows so far
Scanned batch 2: 1,000,000 rows so far
Scanned batch 3: 1,500,000 rows so far
Scanned batch 4: 2,000,000 rows so far
Scanned batch 5: 2,500,000 rows so far
Scanned batch 6: 3,000,000 rows so far
Scanned batch 7: 3,500,000 rows so far
Scanned batch 8: 4,000,000 rows so far
Scanned batch 9: 4,500,000 rows so far
Scanned batch 10: 5,000,000 rows so far
Scanned batch 11: 5,500,000 rows so far
Scanned batch 12: 6,000,000 rows so far
Scanned batch 13: 6,362,620 rows so far

Total transactions: 6,362,620
Fraud transactions: 8,213 (0.1291%)
System-flagged transactions: 16
Step range: 1 to 743
Fraud labels contain only 0 and 1: True
Time steps containing fraud: 741

Fraud-label counts:


,transactions
isFraud,
0,6354407
1,8213



Existing system flag versus dataset fraud label:


,isFlaggedFraud,isFraud,transactions
0,0,0,6354407
1,0,1,8197
2,1,1,16


,transactions,fraud_transactions,fraud_rate_percent
CASH_OUT,2237500,4116,0.183955
PAYMENT,2151495,0,0.000000
CASH_IN,1399284,0,0.000000
TRANSFER,532909,4097,0.768799
DEBIT,41432,0,0.000000



Missing values across the full file:


,missing_values
amount,0
isFlaggedFraud,0
isFraud,0
nameDest,0
nameOrig,0
newbalanceDest,0
newbalanceOrig,0
oldbalanceDest,0
oldbalanceOrg,0
step,0


## What the audit tells us

- `isFraud` is the label to predict. Because fraud is rare, later model comparisons must emphasize precision, recall, F1, PR-AUC, and top-ranked cases—not accuracy alone.
- `nameOrig` and `nameDest` are account identifiers. A transfer creates a directed link from the sender to the receiver.
- `step` is a simulated time index (an hour-like step), not a calendar timestamp. We will use it for chronological validation.
- `isFlaggedFraud` is an existing system rule flag. We will not use it as an input to predict `isFraud`, because it can leak an earlier fraud decision into the model.
- The `newbalance...` fields describe balances after a transaction. If the goal is to score before a transaction completes, they are not available at decision time and must be excluded from predictors.

These checks describe the dataset and modeling assumptions; they do not establish that any real person or transaction is fraudulent.